"""
Selección de modelo — Hotel Booking Demand
==============================================
Este script entrena TODOS los modelos considerados a lo largo del TP sobre la misma
partición temporal (train/test), los compara con las mismas métricas, y confirma
cuál es el modelo a utilizar — con justificación automática basada en los resultados.

Modelos incluidos:
  - Baseline (predice siempre la clase mayoritaria)
  - Regresión Logística
  - Árbol de Decisión
  - Random Forest
  - Gradient Boosting (HistGradientBoostingClassifier)
  - k-NN
  - Naive Bayes (GaussianNB)
  - LDA (Linear Discriminant Analysis)
  - QDA (Quadratic Discriminant Analysis, con regularización)
  - Red Neuronal (MLPClassifier)
  - SVM (aparte, sobre una submuestra — no es comparable en igualdad de condiciones,
    ver la sección correspondiente)

Métrica principal de selección: ROC-AUC sobre el conjunto de test temporal,
consistente con el criterio usado en el resto del TP.
"""

In [1]:
import warnings
warnings.filterwarnings("ignore")

import time
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_STATE = 42

# ============================================================
# 1. CARGA, PARTICIÓN TEMPORAL Y FEATURE ENGINEERING
# ============================================================

In [5]:
base_path = '../data/model_input/static/'
 
X_train = pd.read_csv(base_path + 'X_train.csv')
y_train = pd.read_csv(base_path + 'y_train.csv').values.ravel()
 
X_val = pd.read_csv(base_path + 'X_val.csv')
y_val = pd.read_csv(base_path + 'y_val.csv').values.ravel()
 
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
 

X_train shape: (61176, 23)
X_val shape: (13110, 23)


In [ ]:
"""""
def cargar_y_preparar(path_csv="hotel_bookings.csv"):
    df = pd.read_csv(path_csv)
    df["arrival_date"] = pd.to_datetime(
        df["arrival_date_year"].astype(str) + "-" +
        df["arrival_date_month"] + "-" +
        df["arrival_date_day_of_month"].astype(str),
        format="%Y-%B-%d"
    )
    df_sorted = df.sort_values("arrival_date").reset_index(drop=True)
    n = len(df_sorted)
    corte1 = df_sorted.iloc[int(n * 0.70)]["arrival_date"]
    corte2 = df_sorted.iloc[int(n * 0.85)]["arrival_date"]

    train = df_sorted[df_sorted["arrival_date"] < corte1].copy()
    val = df_sorted[(df_sorted["arrival_date"] >= corte1) & (df_sorted["arrival_date"] < corte2)].copy()
    test = df_sorted[df_sorted["arrival_date"] >= corte2].copy()

    def preparar_features(d):
        d = d.copy()
        d["has_agent"] = d["agent"].notnull().astype(int)
        d["has_company"] = d["company"].notnull().astype(int)
        d["children"] = d["children"].fillna(0)
        d["es_domestico"] = (d["country"] == "PRT").astype(int)
        d["adr"] = d["adr"].clip(lower=0, upper=5000)
        return d

    return preparar_features(train), preparar_features(val), preparar_features(test)


NUM_FEATURES = [
    "lead_time", "arrival_date_year", "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children", "babies",
    "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled",
    "adr", "required_car_parking_spaces", "total_of_special_requests",
    "has_agent", "has_company", "es_domestico",
]
CAT_FEATURES = [
    "hotel", "arrival_date_month", "meal", "market_segment",
    "distribution_channel", "reserved_room_type", "deposit_type", "customer_type",
]


vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]

vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num' 
]

vars_binarias = [
    'is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)
"""

# ============================================================
# 2. EVALUAR UN MODELO: métrica + tiempo, sobre la misma partición
# ============================================================

In [3]:
def evaluar_modelo(nombre, modelo, X_train, y_train, X_test, y_test, preprocessor):
    pipe = Pipeline([("prep", preprocessor), ("clf", modelo)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    t_fit = time.time() - t0

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None

    resultado = {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
        "tiempo_seg": round(t_fit, 2),
    }
    return resultado, pipe

# ============================================================
# 3. FLUJO PRINCIPAL
# ============================================================

# ============================================================
# A PARTIR DE ACÁ: código de nivel superior (pegar en celdas de notebook, en orden)
# ============================================================

In [7]:
vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]

vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num' 
]

vars_binarias = [
    'is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

In [ ]:
"""
print("=" * 70)
print("SELECCIÓN DE MODELO — Hotel Booking Demand")
print("=" * 70)

train, val, test = cargar_y_preparar("hotel_bookings.csv")
print(f"\nPartición temporal -> train={len(train)}, val={len(val)}, test={len(test)}")

target = "is_canceled"
X_train, y_train = train[NUM_FEATURES + CAT_FEATURES], train[target]
X_val, y_val = val[NUM_FEATURES + CAT_FEATURES], val[target]
X_test, y_test = test[NUM_FEATURES + CAT_FEATURES], test[target]

preprocessor = construir_preprocesador()
"""


SELECCIÓN DE MODELO — Hotel Booking Demand


FileNotFoundError: [Errno 2] No such file or directory: 'hotel_bookings.csv'

# ---------- Modelos comparables en igualdad de condiciones ----------
# IMPORTANTE: se comparan contra VALIDACIÓN, no contra test -- test se reserva
# para evaluar UNA SOLA VEZ al ganador ya tuneado, consistente con el resto del TP.

In [8]:
modelos = {
    "Baseline (mayoría)": DummyClassifier(strategy="most_frequent"),
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA (regularizado)": QuadraticDiscriminantAnalysis(reg_param=0.3),
    "Naive Bayes": GaussianNB(),
    "k-NN (k=15)": KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    "Árbol de Decisión": DecisionTreeClassifier(max_depth=12, random_state=RANDOM_STATE, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    "Red Neuronal (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200,
                                         random_state=RANDOM_STATE, early_stopping=True),
}

print("\nEntrenando (con train) y evaluando (con VALIDACIÓN) cada modelo...\n")
resultados = []
for nombre, modelo in modelos.items():
    res, _ = evaluar_modelo(nombre, modelo, X_train, y_train, X_val, y_val, preprocessor)
    resultados.append(res)
    print(f"  {nombre:25s} ROC-AUC={res['roc_auc']:.4f}  Accuracy={res['accuracy']:.4f}  "
          f"F1={res['f1']:.4f}  ({res['tiempo_seg']}s)")

tabla = pd.DataFrame(resultados).sort_values("roc_auc", ascending=False).reset_index(drop=True)


Entrenando (con train) y evaluando (con VALIDACIÓN) cada modelo...

  Baseline (mayoría)        ROC-AUC=0.5000  Accuracy=0.7093  F1=0.0000  (0.8s)
  Regresión Logística       ROC-AUC=0.7907  Accuracy=0.6390  F1=0.5841  (1.88s)
  LDA                       ROC-AUC=0.7871  Accuracy=0.7490  F1=0.5371  (0.53s)
  QDA (regularizado)        ROC-AUC=0.7587  Accuracy=0.6987  F1=0.5688  (0.66s)
  Naive Bayes               ROC-AUC=0.7389  Accuracy=0.7101  F1=0.4381  (0.18s)
  k-NN (k=15)               ROC-AUC=0.7680  Accuracy=0.7130  F1=0.5611  (0.21s)
  Árbol de Decisión         ROC-AUC=0.8341  Accuracy=0.7027  F1=0.6395  (0.81s)
  Random Forest             ROC-AUC=0.8500  Accuracy=0.7167  F1=0.6495  (4.87s)
  Gradient Boosting         ROC-AUC=0.8632  Accuracy=0.7334  F1=0.6654  (4.05s)
  Red Neuronal (MLP)        ROC-AUC=0.8296  Accuracy=0.7751  F1=0.5737  (17.73s)


# ---------- SVM aparte: no comparable en igualdad de condiciones ----------

In [9]:
print("\n" + "-" * 70)
print("SVM: se evalúa aparte, sobre una submuestra de 10.000 filas")
print("(entrenar con las 83.442 filas completas tomaría >10 minutos; no es")
print(" comparable en igualdad de condiciones con el resto de los modelos)")
print("-" * 70)
X_train_sub = X_train.iloc[:10000]
y_train_sub = y_train.iloc[:10000]
res_svm, _ = evaluar_modelo("SVM (submuestra 10k)", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
                              X_train_sub, y_train_sub, X_val, y_val, preprocessor)
print(f"  SVM (10.000 filas): ROC-AUC={res_svm['roc_auc']:.4f}  ({res_svm['tiempo_seg']}s de entrenamiento)")



----------------------------------------------------------------------
SVM: se evalúa aparte, sobre una submuestra de 10.000 filas
(entrenar con las 83.442 filas completas tomaría >10 minutos; no es
 comparable en igualdad de condiciones con el resto de los modelos)
----------------------------------------------------------------------


AttributeError: 'numpy.ndarray' object has no attribute 'iloc'

# ---------- Tabla resumen (sobre VALIDACIÓN) ----------

In [10]:
print("\n" + "=" * 70)
print("TABLA RESUMEN SOBRE VALIDACIÓN (ordenada por ROC-AUC, de mejor a peor)")
print("=" * 70)
print(tabla[["modelo", "roc_auc", "accuracy", "precision", "recall", "f1", "tiempo_seg"]]
      .to_string(index=False))


TABLA RESUMEN SOBRE VALIDACIÓN (ordenada por ROC-AUC, de mejor a peor)
             modelo  roc_auc  accuracy  precision   recall       f1  tiempo_seg
  Gradient Boosting 0.863183  0.733410   0.523817 0.911834 0.665390        4.05
      Random Forest 0.849959  0.716705   0.507148 0.902913 0.649490        4.87
  Árbol de Decisión 0.834080  0.702746   0.493857 0.907111 0.639534        0.81
 Red Neuronal (MLP) 0.829574  0.775057   0.638764 0.520598 0.573659       17.73
Regresión Logística 0.790748  0.638978   0.439102 0.872212 0.584131        1.88
                LDA 0.787148  0.748970   0.578836 0.500918 0.537066        0.53
        k-NN (k=15) 0.768026  0.712967   0.505040 0.631068 0.561064        0.21
 QDA (regularizado) 0.758729  0.698703   0.487007 0.683548 0.568777        0.66
        Naive Bayes 0.738864  0.710069   0.501693 0.388874 0.438137        0.18
 Baseline (mayoría) 0.500000  0.709306   0.000000 0.000000 0.000000        0.80


# ---------- Selección automática del ganador (por validación) ----------

In [ ]:
ganador_inicial = tabla.iloc[0]
nombre_ganador = ganador_inicial["modelo"]
print(f"\nGanador en validación (antes de tunear): {nombre_ganador} "
      f"(ROC-AUC={ganador_inicial['roc_auc']:.4f})")


Ganador en validación (antes de tunear): Gradient Boosting (ROC-AUC=0.8948)


# ---------- Tuning del ganador real (genérico: el que haya ganado, no hardcodeado) ----------

In [ ]:
MODELOS_TUNEABLES = {
    "Gradient Boosting": (HistGradientBoostingClassifier(random_state=RANDOM_STATE), {
        "clf__max_iter": [100, 150, 200], "clf__max_depth": [None, 5, 8, 12],
        "clf__learning_rate": [0.05, 0.08, 0.1, 0.15], "clf__l2_regularization": [0, 0.1, 0.5],
        "clf__max_leaf_nodes": [15, 31, 63],
    }),
    "Random Forest": (RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1), {
        "clf__n_estimators": [100, 200, 300], "clf__max_depth": [10, 15, 20, None],
        "clf__min_samples_leaf": [1, 2, 5], "clf__max_features": ["sqrt", "log2"],
    }),
    "Red Neuronal (MLP)": (MLPClassifier(max_iter=300, random_state=RANDOM_STATE, early_stopping=True), {
        "clf__hidden_layer_sizes": [(32,), (64, 32), (100, 50)],
        "clf__alpha": [0.0001, 0.001, 0.01], "clf__learning_rate_init": [0.001, 0.01],
    }),
}
X_train_val = pd.concat([X_train, X_val], ignore_index=True)
y_train_val = pd.concat([y_train, y_val], ignore_index=True)
test_fold = np.concatenate([np.full(len(X_train), -1), np.full(len(X_val), 0)])
ps = PredefinedSplit(test_fold)

if nombre_ganador in MODELOS_TUNEABLES:
    print("\n" + "-" * 70)
    print(f"Ajustando hiperparámetros del ganador real ({nombre_ganador})...")
    print("-" * 70)
    modelo_base, param_dist = MODELOS_TUNEABLES[nombre_ganador]
    pipe_final = Pipeline([("prep", preprocessor), ("clf", modelo_base)])
    search = RandomizedSearchCV(pipe_final, param_distributions=param_dist, n_iter=12,
                                 scoring="roc_auc", cv=ps, random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train_val, y_train_val)
    modelo_final = search.best_estimator_
    print(f"Mejores hiperparámetros: {search.best_params_}")
    print(f"ROC-AUC en validación (tuneado): {search.best_score_:.4f}")
else:
    print(f"\n({nombre_ganador} no tiene grilla de tuning definida en este script -> se usa sin tunear)")
    modelo_final = Pipeline([("prep", preprocessor), ("clf", modelos[nombre_ganador])])
    modelo_final.fit(X_train_val, y_train_val)


----------------------------------------------------------------------
Ajustando hiperparámetros del ganador real (Gradient Boosting)...
----------------------------------------------------------------------
Mejores hiperparámetros: {'clf__max_leaf_nodes': 31, 'clf__max_iter': 200, 'clf__max_depth': 12, 'clf__learning_rate': 0.08, 'clf__l2_regularization': 0}
ROC-AUC en validación (tuneado): 0.8981


# ---------- Evaluación FINAL: test, una única vez, solo para el ganador ----------

In [ ]:
y_proba_test = modelo_final.predict_proba(X_test)[:, 1]
y_pred_test = modelo_final.predict(X_test)
auc_test = roc_auc_score(y_test, y_proba_test)
acc_test = accuracy_score(y_test, y_pred_test)
print(f"\nEvaluación FINAL sobre TEST (única vez): ROC-AUC={auc_test:.4f}  Accuracy={acc_test:.4f}")


# ---------- Confirmación final con justificación ----------

In [ ]:
print("\n" + "=" * 70)
print("MODELO CONFIRMADO")
print("=" * 70)
baseline_auc = tabla[tabla["modelo"] == "Baseline (mayoría)"]["roc_auc"].values[0]
segundo_lugar = tabla.iloc[1]
print(f"""
Modelo elegido: {nombre_ganador}

Justificación automática, a partir de los resultados de esta corrida:
  1. Mejor ROC-AUC en VALIDACIÓN entre los {len(tabla)} modelos comparados en
 igualdad de condiciones: {ganador_inicial['roc_auc']:.4f}
  2. Supera al baseline trivial por {(ganador_inicial['roc_auc'] - baseline_auc):.4f}
 puntos de ROC-AUC ({baseline_auc:.4f} -> {ganador_inicial['roc_auc']:.4f})
  3. Supera al segundo mejor modelo ({segundo_lugar['modelo']},
 ROC-AUC={segundo_lugar['roc_auc']:.4f}) por
 {(ganador_inicial['roc_auc'] - segundo_lugar['roc_auc']):.4f} puntos
  4. Tras el tuning y la evaluación única sobre test: ROC-AUC={auc_test:.4f}
  5. SVM quedó excluido de la comparación directa por costo computacional
 (no entrena en tiempo razonable con el dataset completo)
""")